# Lab 6 — Building the Knowledge Brain

**Module:** RAG & Embeddings | **Duration:** 90 min | **Switch roles at Part 2**

**Fil Rouge:** DevAssist — AI-Powered Developer Documentation Assistant for TaskFlow

---

## Core Principle

> **RAG works because of attention.** Retrieved passages become context tokens.
> Attention weights shift from parametric memory (hallucination-prone) to retrieved evidence (grounded).

## Pipeline

```
OFFLINE: Docs → Chunk → Embed → Index (ChromaDB)
ONLINE:  Query → Embed → Retrieve top-k → Inject into prompt → Generate grounded answer
```

---
## Setup

In [ ]:
import json, sys, os
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '.')

from utils.generation_utils import generate, is_ollama_available
from utils.chunking_utils import load_and_chunk_corpus, chunk_by_paragraphs
from utils.embedding_utils import cosine_similarity, cosine_similarity_matrix, plot_similarity_heatmap
from utils.retrieval_utils import format_context, query_collection

OLLAMA_OK = is_ollama_available()
print(f"Ollama: {'✓' if OLLAMA_OK else '⚠ unavailable'}")

# Try loading sentence-transformers
SBERT_OK = False
try:
    from sentence_transformers import SentenceTransformer
    SBERT_OK = True
    print('✓ sentence-transformers available')
except ImportError:
    print('⚠ sentence-transformers unavailable — using precomputed')

CHROMA_OK = False
try:
    import chromadb
    CHROMA_OK = True
    print('✓ ChromaDB available')
except ImportError:
    print('⚠ ChromaDB unavailable — using precomputed')

with open('data/precomputed_outputs.json') as f:
    PRECOMPUTED = json.load(f)
print('✓ Precomputed data loaded')

---
# PART 1 — EMBEDDING EXPLORATION (20 min)

**🟢 Driver starts here.**

---
## Exercise 1A: Semantic Similarity Matrix

In [ ]:
queries = [
    "How do I install the project?",
    "What authentication method does the API use?",
    "How do I run the test suite?",
]

chunks = [
    "To install the project, clone the repository and run pip install -r requirements.txt.",
    "The API uses OAuth2 bearer tokens for authentication. Set the token in the Authorization header.",
    "Run the test suite with: python -m pytest tests/ -v",
    "The project was started in 2022 by the open-source community.",
    "For production deployment, use Docker with the provided Dockerfile.",
    "Authentication tokens expire after 1 hour. Use the refresh endpoint to obtain a new token.",
]

if SBERT_OK:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    query_emb = model.encode(queries)
    chunk_emb = model.encode(chunks)
    sim_matrix = cosine_similarity_matrix(query_emb, chunk_emb)
else:
    sim_matrix = np.array(PRECOMPUTED['embedding_exploration']['similarity_matrix'])

plot_similarity_heatmap(sim_matrix, queries, chunks, title='Query-Chunk Similarity Matrix')

### 1A Analysis

**Q1: For each query, which chunk has the highest similarity? Does this match intuition?**

TODO

**Q2: Does the auth query also match the token expiry chunk (#6)? Why?**

TODO

**Q3: If top_k=2, which chunks would be retrieved for each query?**

TODO

---
## Exercise 1B: Keyword vs. Semantic Retrieval

In [ ]:
# "I can't log in" vs auth chunk vs logging chunk
query_natural = "I can't log in to the system"
relevant_chunk = "The API uses OAuth2 bearer tokens for authentication. Set the token in the Authorization header."
irrelevant_chunk = "The project's logging system writes to stdout by default."

if SBERT_OK:
    embs = model.encode([query_natural, relevant_chunk, irrelevant_chunk])
    nat_to_rel = cosine_similarity(embs[0], embs[1])
    nat_to_irr = cosine_similarity(embs[0], embs[2])
    print(f'"I can\'t log in" ↔ auth chunk:    {nat_to_rel:.3f}')
    print(f'"I can\'t log in" ↔ logging chunk: {nat_to_irr:.3f}')
    print(f'\nSemantic retrieval correctly identifies auth > logging!')
    print(f'Keyword matching would be confused by "log" in both.')
else:
    kv = PRECOMPUTED['embedding_exploration']['keyword_vs_semantic']
    print(f'"I can\'t log in" ↔ auth chunk:    {kv["natural_to_relevant"]:.3f}')
    print(f'"I can\'t log in" ↔ logging chunk: {kv["natural_to_irrelevant"]:.3f}')
    print(f'\nSemantic retrieval correctly identifies auth > logging!')

### 1B Analysis

**Q1: Does semantic search correctly retrieve the auth chunk for "I can't log in"?**

TODO

**Q2: Would keyword matching confuse "log in" with "logging"?**

TODO

**Q3: Connection to Module 2 — how does the embedding model's learned similarity relate to attention?**

TODO

---
# PART 2 — BUILD THE RAG PIPELINE (45 min)

**🔄 Switch driver/navigator.**

---
## Step 1: Chunk the Corpus

In [ ]:
# Load and chunk all TaskFlow documentation
all_chunks = load_and_chunk_corpus('corpus/docs', strategy='sections', min_length=80)

print(f'Total chunks: {len(all_chunks)}')
print(f'\nChunks per file:')
from collections import Counter
for fname, count in Counter(c['source_file'] for c in all_chunks).items():
    print(f'  {fname}: {count} chunks')

print(f'\nSample chunk:')
print(f'  Source: {all_chunks[0]["source_file"]} | Section: {all_chunks[0]["section"]}')
print(f'  Text: {all_chunks[0]["text"][:200]}...')

---
## Step 2: Embed & Index in ChromaDB

In [ ]:
if SBERT_OK and CHROMA_OK:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    client = chromadb.Client()
    
    # Delete existing collection if re-running
    try:
        client.delete_collection('taskflow_docs')
    except Exception:
        pass
    
    collection = client.create_collection(
        name='taskflow_docs',
        metadata={'hnsw:space': 'cosine'}
    )
    
    chunk_texts = [c['text'] for c in all_chunks]
    chunk_ids = [f'chunk_{i}' for i in range(len(all_chunks))]
    chunk_metas = [{'source_file': c['source_file'], 'section': c.get('section', '')}
                   for c in all_chunks]
    
    # Embed
    print('Computing embeddings...')
    chunk_embeddings = embedding_model.encode(chunk_texts).tolist()
    
    # Index
    collection.add(
        ids=chunk_ids,
        documents=chunk_texts,
        embeddings=chunk_embeddings,
        metadatas=chunk_metas
    )
    print(f'✓ Indexed {collection.count()} chunks in ChromaDB')
    
    # Verification query
    test_results = query_collection(collection, embedding_model, 'How do I install TaskFlow?', top_k=3)
    print(f'\nVerification: "How do I install TaskFlow?"')
    for i, r in enumerate(test_results):
        print(f'  [{i+1}] {r["source"]} (sim: {r["similarity"]:.3f})')
        print(f'      {r["text"][:120]}...')
else:
    print('⚠ SBERT or ChromaDB unavailable.')
    print('  Using precomputed outputs for remaining exercises.')
    embedding_model = None
    collection = None

---
## Step 3: Build the RAG Prompt & Generate

In [ ]:
def rag_query(question, collection, embedding_model, top_k=3):
    """
    Complete RAG pipeline: retrieve → inject → generate.
    """
    # Retrieve
    passages = query_collection(collection, embedding_model, question, top_k=top_k)
    context_block = format_context(passages)
    
    # RAG Prompt (Contract Framework)
    prompt = f"""You are a helpful assistant that answers questions about the TaskFlow project.

INSTRUCTIONS:
- Answer ONLY using the information in the Context sections below.
- If the context doesn't contain the answer, say: "The documentation does not contain this information."
- Cite sources using [Source N] after each claim.
- Be concise and accurate.

CONTEXT:
{context_block}

QUESTION: {question}

ANSWER (with citations):"""
    
    answer = generate(prompt)
    
    return {
        'question': question,
        'passages': passages,
        'prompt': prompt,
        'answer': answer
    }


def no_rag_query(question):
    """Baseline: ask without context."""
    prompt = f"""Answer the following question about a project called TaskFlow.

QUESTION: {question}

ANSWER:"""
    return generate(prompt)


# Test it
if OLLAMA_OK and collection is not None:
    test = rag_query('How do I authenticate with the TaskFlow API?', collection, embedding_model)
    print(f'Question: {test["question"]}')
    print(f'\nRetrieved passages:')
    for i, p in enumerate(test['passages']):
        print(f'  [{i+1}] {p["source"]} (sim: {p["similarity"]:.3f})')
    print(f'\nRAG Answer:\n{test["answer"]}')
    print(f'\n--- No-RAG Baseline ---')
    print(no_rag_query('How do I authenticate with the TaskFlow API?'))
else:
    print('Using precomputed RAG outputs:')
    print(f'\nRAG: {PRECOMPUTED["rag_answers"]["q1_auth"]["rag"]}')
    print(f'\nNo-RAG: {PRECOMPUTED["rag_answers"]["q1_auth"]["no_rag"]}')

---
# PART 3 — RAG vs. NO-RAG EVALUATION (15 min)

Run 5 questions with and without RAG. Score each using the rubric.

In [ ]:
with open('tests/test_questions.json') as f:
    test_questions = json.load(f)

eval_questions = test_questions[:5]
evaluation_results = []

for q in eval_questions:
    question = q['question']
    print(f'\n{"="*60}')
    print(f'Q{q["id"]}: {question}')
    print(f'Ground truth: {q["ground_truth_answer"]}')
    print(f'{"="*60}')
    
    if OLLAMA_OK and collection is not None:
        result = rag_query(question, collection, embedding_model)
        rag_answer = result['answer']
        no_rag_answer = no_rag_query(question)
        sources = [p['source'] for p in result['passages']]
    else:
        # Use precomputed for available questions
        key_map = {1: 'q1_auth', 2: 'q2_states', 3: 'q3_install', 5: 'q5_state_skip'}
        key = key_map.get(q['id'])
        if key:
            rag_answer = PRECOMPUTED['rag_answers'][key]['rag']
            no_rag_answer = PRECOMPUTED['rag_answers'][key]['no_rag']
        else:
            rag_answer = '[precomputed not available for this question]'
            no_rag_answer = '[precomputed not available for this question]'
        sources = [q['source_file']]
    
    print(f'\nRAG Answer: {rag_answer[:300]}')
    print(f'\nNo-RAG Answer: {no_rag_answer[:300]}')
    
    evaluation_results.append({
        'id': q['id'],
        'question': question,
        'ground_truth': q['ground_truth_answer'],
        'rag_answer': rag_answer,
        'no_rag_answer': no_rag_answer,
        'sources': sources,
        'rag_scores': {'accuracy': None, 'groundedness': None, 'citation': None},
        'no_rag_scores': {'accuracy': None, 'groundedness': None, 'citation': None},
    })

### Score Each Answer

Use the rubric:
- **Accuracy** (0-2): 0=wrong, 1=partial, 2=correct
- **Groundedness** (0-2): 0=hallucinated, 1=mixed, 2=grounded
- **Citation** (0-2): 0=none/wrong, 1=partial, 2=correct

In [ ]:
# ============================================================
# TODO: Score each question by editing the scores below.
# After scoring, run this cell to see the comparison.
# ============================================================

# Example scoring (replace with your actual scores):
# evaluation_results[0]['rag_scores'] = {'accuracy': 2, 'groundedness': 2, 'citation': 2}
# evaluation_results[0]['no_rag_scores'] = {'accuracy': 0, 'groundedness': 0, 'citation': 0}

# TODO: Score all 5 questions

# Print comparison table
print(f'{"Q":>3}  {"RAG Acc":>8}  {"NoRAG Acc":>10}  {"RAG Gnd":>8}  {"NoRAG Gnd":>10}  {"RAG Cit":>8}')
print('-' * 65)
for r in evaluation_results:
    ra = r['rag_scores']['accuracy'] or '?'
    na = r['no_rag_scores']['accuracy'] or '?'
    rg = r['rag_scores']['groundedness'] or '?'
    ng = r['no_rag_scores']['groundedness'] or '?'
    rc = r['rag_scores']['citation'] or '?'
    print(f'{r["id"]:>3}  {ra:>8}  {na:>10}  {rg:>8}  {ng:>10}  {rc:>8}')

### Evaluation Analysis

**Q1: On how many questions did RAG improve accuracy?**

TODO

**Q2: Where did RAG fail or underperform?**

TODO

**Q3: How did the no-RAG model handle TaskFlow-specific questions?**

TODO

**Q4: Were RAG citations accurate?**

TODO

**Q5: Mechanistic reflection — why does context injection reduce hallucination?**

TODO

---
# WRAP-UP

## Key Takeaways

1. TODO
2. TODO
3. TODO

## Checklist

- ✅ Similarity heatmap generated (Part 1)
- ✅ Keyword vs. semantic analysis complete
- ✅ RAG pipeline functional: chunk → embed → index → retrieve → generate
- ✅ 5 questions scored RAG vs. no-RAG
- ✅ `rag_evaluation.md` completed
- ✅ `git add -A && git commit -m 'Lab 6 complete' && git push`

## Portfolio Entry

**Use Case #6:** RAG Q&A over TaskFlow docs — hallucination comparison data.

## Preview: Lab 7

In **Lab 7 — Attack & Defend**, you'll red-team this RAG pipeline:
prompt injection via malicious documents, data exfiltration, and jailbreaks.
Then you'll harden the system against these attacks.

---
*Lab 6 of 8 — DevAssist / TaskFlow Lab Series*